In [ ]:
library(here)
library(maplet)
library(dplyr)
library(purrr)
library(fgsea)

# set repo path
repo <- here()
renv::activate(project = repo)

# Pathway Analysis

## Pathway Data Preparation

In [ ]:
# load maplet object
D <- readRDS(here('data', 'preprocessed_venous_metabolon.RDS'))

In [ ]:
# store pathway information separately
rowdata <- D %>% rowData() %>% as.data.frame()

In [ ]:
# create a named list of pathway information
sub_pathways <- D %>% rowData() %>% as.data.frame() %>% 
    tibble::rownames_to_column('var') %>% # set rownames as a column
    group_by(SUB_PATHWAY) %>% # group by pathway
    summarise(metabolites = list(var)) %>% # convert to a list
    tibble::deframe() # convert to named list (pathway -> metabolites)

## MGSEA

In [ ]:
# specify significance level
msea_sig_level <- 0.05

### GLS 6`

In [ ]:
# load associations
df_stats_gls <- readxl::read_excel(here('outputs', 'RVGLOB6n_univariate_test.xlsx'), sheet = 1)

# prepare ranking vector for metabolites
metabolite_ranks <- df_stats_gls %>%
  arrange(desc(statistic)) %>%   # Arrange by effect size
  select(var, statistic) %>%     # Keep only metabolite name and effect size
  tibble::deframe()                     # Convert to named vector

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 8)

# set seed for reproducibility
set.seed(123)

# compute fgsea 
fgsea_results_gls <- fgsea(
  pathways = sub_pathways,
  stats = metabolite_ranks,
  nperm = 100000 # or set perm num 
)

# print significant pathways
fgsea_results_gls %>% arrange(padj) %>% filter(padj < msea_sig_level) %>% pull(pathway)

### FAC

In [ ]:
# load associations
df_stats_fac <- readxl::read_excel(here('outputs', 'RVFACn_univariate_test.xlsx'), sheet = 1)

# prepare ranking vector for metabolites
metabolite_ranks <- df_stats_fac %>%
  arrange(desc(statistic)) %>%   # Arrange by effect size
  select(var, statistic) %>%     # Keep only metabolite name and effect size
  tibble::deframe()                     # Convert to named vector

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 8)

# set seed for reproducibility
set.seed(123)

# compute fgsea 
fgsea_results_fac <- fgsea(
  pathways = sub_pathways,
  stats = metabolite_ranks,
  nperm = 100000 # or set perm num 
)

# print significant pathways
fgsea_results_fac %>% arrange(padj) %>% filter(padj < msea_sig_level) %>% pull(pathway)

### RVEF

In [ ]:
# load associations
df_stats_rvef <- readxl::read_excel(here('outputs', 'mri_RVEF_univariate_test.xlsx'), sheet = 1)

# prepare ranking vector for metabolites
metabolite_ranks <- df_stats_rvef %>%
  arrange(desc(statistic)) %>%   # Arrange by effect size
  select(var, statistic) %>%     # Keep only metabolite name and effect size
  tibble::deframe()                     # Convert to named vector

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 8)

# set seed for reproducibility
set.seed(123)

# compute fgsea 
fgsea_results_rvef <- fgsea(
  pathways = sub_pathways,
  stats = metabolite_ranks,
  nperm = 100000 # or set perm num 
)

# print significant pathways
fgsea_results_rvef %>% arrange(padj) %>% filter(padj < msea_sig_level) %>% pull(pathway)

## Aggregation

In [ ]:
# Define confounders as a string
confounders <- "age + SEX + bmi + f_wnowt"

### GLS 6`

In [ ]:
# specify variable of interest 
var_of_interest <- "RVGLOB6n"

# Construct the formula dynamically
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# aggregation based approach
D_aggmean <- D %>% 
    mt_pre_trans_scale() %>% # scale metabolite abundances (mean-center, unit-variance)
    mt_modify_agg_pathways(pw_col = "SUB_PATHWAY", method = "aggmean") %>% # mean-aggregate metabolites by pathway
    mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "aggmean_met")) %>% # univariate lm with RV
    # add multiple testing correction
    mt_post_multtest(stat_name = paste(var_of_interest, "aggmean_met"), method = "BH") %>%
    # add stats logging
    mt_reporting_stats(stat_name = paste(var_of_interest, "aggmean_met"), stat_filter = p.adj < 0.05) %>% 
    {.}


In [ ]:
# write statistical results to file
D_aggmean <- D_aggmean %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_mean_aggregation_pathway_results.xlsx")))

### FAC

In [ ]:
# specify variable of interest 
var_of_interest <- "RVFACn"

# Construct the formula dynamically
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# aggregation based approach
D_aggmean <- D %>% 
    mt_pre_trans_scale() %>% # scale metabolite abundances (mean-center, unit-variance)
    mt_modify_agg_pathways(pw_col = "SUB_PATHWAY", method = "aggmean") %>% # mean-aggregate metabolites by pathway
    mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "aggmean_met")) %>% # univariate lm with RV
    # add multiple testing correction
    mt_post_multtest(stat_name = paste(var_of_interest, "aggmean_met"), method = "BH") %>%
    # add stats logging
    mt_reporting_stats(stat_name = paste(var_of_interest, "aggmean_met"), stat_filter = p.adj < 0.05) %>% 
    {.}


In [ ]:
# write statistical results to file
D_aggmean <- D_aggmean %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_mean_aggregation_pathway_results.xlsx")))

### RVEF

In [ ]:
# specify variable of interest 
var_of_interest <- "mri_RVEF"

# Construct the formula dynamically
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# aggregation based approach
D_aggmean <- D %>% 
    mt_pre_trans_scale() %>% # scale metabolite abundances (mean-center, unit-variance)
    mt_modify_agg_pathways(pw_col = "SUB_PATHWAY", method = "aggmean") %>% # mean-aggregate metabolites by pathway
    mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "aggmean_met")) %>% # univariate lm with RV
    # add multiple testing correction
    mt_post_multtest(stat_name = paste(var_of_interest, "aggmean_met"), method = "BH") %>%
    # add stats logging
    mt_reporting_stats(stat_name = paste(var_of_interest, "aggmean_met"), stat_filter = p.adj < 0.05) %>% 
    {.}

In [ ]:
# write statistical results to file
D_aggmean <- D_aggmean %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_mean_aggregation_pathway_results.xlsx")))

## Sensitivity Analysis Using Eigenmet

### GLS 6`

In [ ]:
# specify variable of interest 
var_of_interest <- "RVGLOB6n"

# Construct the formula dynamically
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# aggregation based approach
D_eigen <- D %>% 
    mt_pre_trans_scale() %>% # scale metabolite abundances (mean-center, unit-variance)
    mt_modify_agg_pathways(pw_col = "SUB_PATHWAY", method = "eigen") %>% # mean-aggregate metabolites by pathway
    mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "eigen_met")) %>% # univariate lm with RV
    # add multiple testing correction
    mt_post_multtest(stat_name = paste(var_of_interest, "eigen_met"), method = "BH") %>%
    # add stats logging
    mt_reporting_stats(stat_name = paste(var_of_interest, "eigen_met"), stat_filter = p.adj < 0.05) %>% 
    {.}

In [ ]:
# write statistical results to file
D_eigen <- D_eigen %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_eigen_aggregation_pathway_results.xlsx")))

### FAC

In [ ]:
# specify variable of interest 
var_of_interest <- "RVFACn"

# Construct the formula dynamically
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# aggregation based approach
D_eigen <- D %>% 
    mt_pre_trans_scale() %>% # scale metabolite abundances (mean-center, unit-variance)
    mt_modify_agg_pathways(pw_col = "SUB_PATHWAY", method = "eigen") %>% # mean-aggregate metabolites by pathway
    mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "eigen_met")) %>% # univariate lm with RV
    # add multiple testing correction
    mt_post_multtest(stat_name = paste(var_of_interest, "eigen_met"), method = "BH") %>%
    # add stats logging
    mt_reporting_stats(stat_name = paste(var_of_interest, "eigen_met"), stat_filter = p.adj < 0.05) %>% 
    {.}

In [ ]:
# write statistical results to file
D_eigen <- D_eigen %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_eigen_aggregation_pathway_results.xlsx")))

### RVEF

In [ ]:
# specify variable of interest 
var_of_interest <- "mri_RVEF"

# Construct the formula dynamically
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# aggregation based approach
D_eigen <- D %>% 
    mt_pre_trans_scale() %>% # scale metabolite abundances (mean-center, unit-variance)
    mt_modify_agg_pathways(pw_col = "SUB_PATHWAY", method = "eigen") %>% # mean-aggregate metabolites by pathway
    mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "eigen_met")) %>% # univariate lm with RV
    # add multiple testing correction
    mt_post_multtest(stat_name = paste(var_of_interest, "eigen_met"), method = "BH") %>%
    # add stats logging
    mt_reporting_stats(stat_name = paste(var_of_interest, "eigen_met"), stat_filter = p.adj < 0.05) %>% 
    {.}

In [ ]:
# write statistical results to file
D_eigen <- D_eigen %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_eigen_aggregation_pathway_results.xlsx")))